# Training model to recognize people name

Ce notebook utilise multinerd data en français et re-entraîne un model Camembert en utilisant du transfer learning.

J'ai ré-appris les 2 dernières couches, et obtenu une bonne cross-validation error et accuracy.

Je préfère train les "feed-forward" layers d'abord. <br/>
Les query, key, value layers d'une attention head donne beaucoup de flexibilité au modèle, et il est donc facile d'overfitter un modèle aussi flexible. <br/>
Ce sont aussi ces query, key et value layers qui contiennent une grosse partie de la compréhension du language (à quoi doit-on faire attention pour prédire). <br/>
Donc je préfère apprendre les feed-forward layers (moins overfittable et menant à la prédiction) pour avoir un modèle capable de prédire mon output (si un mot est un nom de personn), puis entraîner les query, key et value layers (qui vont comprendre quelle partie du language est importante pour notre output)

En ayant entraîné les 2 dernières layers, j'ai une accuracy de 99.8%, qui sera suffisante pour la majorité des use-cases.wbr>

In [1]:
import csv
import numpy as npi
import torch
import transformers

In [2]:
with open("../data/raw/multi_nerd/train_fr.tsv") as f:
    rows = list(line.strip().split("\t") for line in f)

In [3]:
def make_labelled_sentences(tagged_words):
    # Joining words until we meet a dot
    # Word's label is 1 if 'PER' is in its tag
    X = []
    y = []

    this_word = []
    this_labels = []
    for tagged_word in tagged_words:
        if len(tagged_word) < 3:
            # not a tagged word
            continue
        word = tagged_word[1]
        tag = tagged_word[2]

        if word == '.':
            X.append(this_word)
            y.append(this_labels)

            this_word = []
            this_labels = []
        else:
            this_word.append(word)
            this_labels.append(1 * tag.endswith("PER"))

    return X, y

In [4]:
sentences, labels = make_labelled_sentences(rows)

In [5]:
from sklearn.model_selection import train_test_split

sentences_left, sentences_test, labels_left, labels_test = train_test_split(
    sentences,
    labels,
    test_size=0.2,
    random_state=42,
)

sentences_train, sentences_dev, labels_train, labels_dev = train_test_split(
    sentences_left,
    labels_left,
    test_size=0.2,
    random_state=42,
)

# Loading tokenizer

In [6]:
from datasets import load_dataset
from transformers import (
    AutoModelForTokenClassification,
    AutoTokenizer,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)
from peft import get_peft_config, PeftModel, PeftConfig, get_peft_model, LoraConfig, TaskType
import evaluate
import torch
import numpy as np

model_checkpoint = "almanach/camembert-base"
lr = 1e-3
batch_size = 16
num_epochs = 10

In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, add_prefix_space=True)

In [8]:
def tokenize_and_align_labels(sentences, ner_tags):
    tokenized_inputs = tokenizer(
        sentences,
        truncation=True,
        is_split_into_words=True,
    )
    labels = []
    for i, label in enumerate(ner_tags):
        word_ids = tokenized_inputs.word_ids(batch_index=i)  # Map tokens to their respective word.
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:  # Set the special tokens to -100.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:  # Only label the first token of a given word.
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)

            previous_word_idx = word_idx

        labels.append(label_ids)

    tokenized_inputs["labels"] = labels

    return tokenized_inputs

In [9]:
tokenized_train = tokenize_and_align_labels(sentences_train, labels_train)

In [10]:
tokenized_dev = tokenize_and_align_labels(sentences_dev, labels_dev)

In [11]:
tokenized_test = tokenize_and_align_labels(sentences_test, labels_test)

In [12]:
from datasets import Dataset

dataset_train = Dataset.from_dict(tokenized_train)
dataset_dev = Dataset.from_dict(tokenized_dev)
dataset_test = Dataset.from_dict(tokenized_test)

In [13]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [14]:
import numpy as np
import evaluate

seqeval = evaluate.load("seqeval")

labels = [0, 1]
label_list = ["0", "1"]

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=2)

In [ ]:
model = model.to("cuda")

In [ ]:
print(model)

In [ ]:
for name, param in model.base_model.named_parameters():
  print(name)

## Freeze all but last layer

In [ ]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False

for name, param in model.base_model.named_parameters():
  if (
        any(layer_name in name for layer_name in ["layer.11"])
        and any(layer_type in name for layer_type in ["weight", "bias"])
        and "attention" not in name
    ):
        param.requires_grad = True

In [ ]:
model = model.to("cuda")

In [ ]:
batch_size = 16

training_args = TrainingArguments(
    output_dir="ner_unfreeze_last_layer",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## Freeze all but 2 last layers

In [15]:
model = AutoModelForTokenClassification.from_pretrained("ner_unfreeze_2_last_layers/checkpoint-11236/")

In [ ]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False
    
for name, param in model.base_model.named_parameters():
  if (
        any(layer_name in name for layer_name in ["layer.10", "layer.11"])
        and any(layer_type in name for layer_type in ["weight", "bias"])
        and "attention" not in name
    ):
        param.requires_grad = True

In [ ]:
batch_size = 16

training_args = TrainingArguments(
    output_dir="ner_unfreeze_2_last_layers",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

## UnFreeze last attention layer

In [16]:
model = AutoModelForTokenClassification.from_pretrained("ner_unfreeze_last_layer/checkpoint-11236/")

In [19]:
for name, param in model.base_model.named_parameters():
  param.requires_grad = False
    
for name, param in model.base_model.named_parameters():
  if (
        any(layer_name in name for layer_name in ["layer.10.intermediate", "layer.10.output", "layer.11"])
    ):
        param.requires_grad = True

In [20]:
batch_size = 16

training_args = TrainingArguments(
    output_dir="ner_unfreeze_last_attention_layer",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=2,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset_train,
    eval_dataset=dataset_dev,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/accelerate/accelerator.py:432: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,0.005700,0.003858,0.000000,0.000000,0.000000,0.998710
2,0.004000,0.003000,0.000000,0.000000,0.000000,0.999037


/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 0 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: 1 seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/arnaud/.virtualenvs/nlp_esgi/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 due to no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifi

TrainOutput(global_step=11236, training_loss=0.00553573647844872, metrics={'train_runtime': 1044.1513, 'train_samples_per_second': 172.155, 'train_steps_per_second': 10.761, 'total_flos': 6223532469153000.0, 'train_loss': 0.00553573647844872, 'epoch': 2.0})